In [1]:
import pandas as pd
import numpy as np
import os
import joblib as jb
from datetime import datetime
from src.raw_preprocessing import (
    school_holidays_preprocess, 
    conso_preprocess, 
    public_holidays_preprocess)

# Multiple steps

## Conso (py pipeline)

In [2]:
conso2 = conso_preprocess("data/conso/")
conso2

,Date,Heures,Consommation
0,2021-01-01,00:00,67156.0
2,2021-01-01,00:30,66483.0
4,2021-01-01,01:00,64390.0
6,2021-01-01,01:30,64223.0
8,2021-01-01,02:00,63689.0
...,...,...,...
140249,2024-12-31,21:30,63946.0
140251,2024-12-31,22:00,63063.0
140253,2024-12-31,22:30,63816.0
140255,2024-12-31,23:00,65437.0


## School holidays (py pipeline)

In [3]:
holidays2 = school_holidays_preprocess(conso2, PATH_HOLIDAYS="data/calendar/", PATH_ARTIFACTS="artifacts/py_artifacts/")
holidays2

(Dictionnary successfully loaded)


,Date,Zone_A,Zone_B,Zone_C,Vacances de la Toussaint,Vacances de Noël,Vacances d'Hiver,Vacances de Printemps,Vacances d'Été
0,2021-01-01,True,True,True,0,1,0,0,0
2,2021-01-01,True,True,True,0,1,0,0,0
4,2021-01-01,True,True,True,0,1,0,0,0
6,2021-01-01,True,True,True,0,1,0,0,0
8,2021-01-01,True,True,True,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...
140249,2024-12-31,False,False,False,0,0,0,0,0
140251,2024-12-31,False,False,False,0,0,0,0,0
140253,2024-12-31,False,False,False,0,0,0,0,0
140255,2024-12-31,False,False,False,0,0,0,0,0


## Merging of `conso` and `holidays2`

In [4]:
# We merge them with pd.concat instead of pd.merge beacause the axis and the sizes are identical.
conso2 = pd.concat([conso2, holidays2.drop("Date", axis=1)], axis=1)

In [5]:
conso2.head()

,Date,Heures,Consommation,Zone_A,Zone_B,Zone_C,Vacances de la Toussaint,Vacances de Noël,Vacances d'Hiver,Vacances de Printemps,Vacances d'Été
0,2021-01-01,00:00,67156.0,True,True,True,0,1,0,0,0
2,2021-01-01,00:30,66483.0,True,True,True,0,1,0,0,0
4,2021-01-01,01:00,64390.0,True,True,True,0,1,0,0,0
6,2021-01-01,01:30,64223.0,True,True,True,0,1,0,0,0
8,2021-01-01,02:00,63689.0,True,True,True,0,1,0,0,0


## Public Holidays (pipeline)

In [6]:
public_holidays_preprocess(conso2, "data/calendar/")
conso2

,Date,Heures,Consommation,Zone_A,Zone_B,Zone_C,Vacances de la Toussaint,Vacances de Noël,Vacances d'Hiver,Vacances de Printemps,Vacances d'Été,public_holidays
0,2021-01-01,00:00,67156.0,True,True,True,0,1,0,0,0,1
2,2021-01-01,00:30,66483.0,True,True,True,0,1,0,0,0,1
4,2021-01-01,01:00,64390.0,True,True,True,0,1,0,0,0,1
6,2021-01-01,01:30,64223.0,True,True,True,0,1,0,0,0,1
8,2021-01-01,02:00,63689.0,True,True,True,0,1,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...
140249,2024-12-31,21:30,63946.0,False,False,False,0,0,0,0,0,0
140251,2024-12-31,22:00,63063.0,False,False,False,0,0,0,0,0,0
140253,2024-12-31,22:30,63816.0,False,False,False,0,0,0,0,0,0
140255,2024-12-31,23:00,65437.0,False,False,False,0,0,0,0,0,0


# All in one

In [4]:
conso = None
if os.path.isfile("artifacts/data_artifacts/conso.pkl"):
    conso = jb.load("artifacts/data_artifacts/conso.pkl")
    print("conso dataset succesfully loaded")
else : 
    conso = conso_preprocess("data/conso/")
    holidays2 = school_holidays_preprocess(conso, PATH_HOLIDAYS="data/calendar/", PATH_ARTIFACTS="artifacts/py_artifacts/")
    conso = pd.concat([conso, holidays2.drop("Date", axis=1)], axis=1)
    public_holidays_preprocess(conso, "data/calendar/")
    jb.dump(conso, "artifacts/data_artifacts/conso.pkl")
    print("conso dataset succesfully created")


(Dictionnary successfully loaded)
conso dataset succesfully created


In [8]:
print(conso["Heures"].iloc[0])
print(conso["Date"].iloc[0])

00:00:00
2021-01-01


## Weather  
Source : https://www.data.gouv.fr/datasets/donnees-climatologiques-de-base-horaires

In [24]:
meteo75 = pd.read_csv("data/weather/H_75_previous-2020-2024.csv", sep =';')

In [25]:
meteo75

,NUM_POSTE,NOM_USUEL,LAT,LON,ALTI,AAAAMMJJHH,RR1,QRR1,DRR1,QDRR1,...,INS2,QINS2,TLAGON,QTLAGON,TVEGETAUX,QTVEGETAUX,ECOULEMENT,QECOULEMENT,STATUS_FXI3S,STATUS_DXI3S
0,75106001,LUXEMBOURG,48.844833,2.338500,50,2020010100,0.0,1.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,75106001,LUXEMBOURG,48.844833,2.338500,50,2020010101,0.0,1.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,75106001,LUXEMBOURG,48.844833,2.338500,50,2020010102,0.0,1.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,75106001,LUXEMBOURG,48.844833,2.338500,50,2020010103,0.0,1.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,75106001,LUXEMBOURG,48.844833,2.338500,50,2020010104,0.0,1.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261855,75116008,LONGCHAMP,48.854833,2.233667,27,2024123119,0.0,1.0,NaN,NaN,...,0.0,9.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.0
261856,75116008,LONGCHAMP,48.854833,2.233667,27,2024123120,0.0,1.0,NaN,NaN,...,0.0,9.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.0
261857,75116008,LONGCHAMP,48.854833,2.233667,27,2024123121,0.0,1.0,NaN,NaN,...,0.0,9.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.0
261858,75116008,LONGCHAMP,48.854833,2.233667,27,2024123122,0.0,1.0,NaN,NaN,...,0.0,9.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.0


In [26]:
cols_to_keep = [
    'NUM_POSTE',        # station ID : identifies the city
    'LAT', 'LON',       # coordinates: used to query Open-Meteo
    'AAAAMMJJHH',       # timestamp
    # Usefull features that we can found with the API of open-meteo
    'T',                # temperature_2m        
    'U',                # relative_humidity_2m  
    'FF',               # wind_speed_10m        
    'PMER',             # pressure_msl          
    'RR1',              # precipitation         
]

meteo75 = meteo75[cols_to_keep]

In [27]:
meteo75

,NUM_POSTE,LAT,LON,AAAAMMJJHH,T,U,FF,PMER,RR1
0,75106001,48.844833,2.338500,2020010100,1.6,NaN,NaN,NaN,0.0
1,75106001,48.844833,2.338500,2020010101,0.9,NaN,NaN,NaN,0.0
2,75106001,48.844833,2.338500,2020010102,0.1,NaN,NaN,NaN,0.0
3,75106001,48.844833,2.338500,2020010103,0.2,NaN,NaN,NaN,0.0
4,75106001,48.844833,2.338500,2020010104,0.6,NaN,NaN,NaN,0.0
...,...,...,...,...,...,...,...,...,...
261855,75116008,48.854833,2.233667,2024123119,4.9,94.0,5.0,NaN,0.0
261856,75116008,48.854833,2.233667,2024123120,5.3,93.0,5.1,NaN,0.0
261857,75116008,48.854833,2.233667,2024123121,5.6,93.0,4.4,NaN,0.0
261858,75116008,48.854833,2.233667,2024123122,5.6,95.0,5.0,NaN,0.0


#### Let's separate the date and hours by creating 2 new columns and make them compatible with the main dataset `conso`

In [28]:
meteo75["Date"] = meteo75["AAAAMMJJHH"].apply(lambda x : str(str(x)[:8]))
meteo75["Date"] = meteo75["Date"].apply(lambda x : datetime.strptime(x, '%Y%m%d').date())

meteo75["Heures"] = meteo75["AAAAMMJJHH"].apply(lambda x : str(str(x)[8:]) + ':00')
meteo75["Heures"] = meteo75["Heures"].apply(lambda x : datetime.strptime(x, '%H:%M').time())

meteo75 = meteo75.drop("AAAAMMJJHH", axis=1)

In [29]:
meteo75

,NUM_POSTE,LAT,LON,T,U,FF,PMER,RR1,Date,Heures
0,75106001,48.844833,2.338500,1.6,NaN,NaN,NaN,0.0,2020-01-01,00:00:00
1,75106001,48.844833,2.338500,0.9,NaN,NaN,NaN,0.0,2020-01-01,01:00:00
2,75106001,48.844833,2.338500,0.1,NaN,NaN,NaN,0.0,2020-01-01,02:00:00
3,75106001,48.844833,2.338500,0.2,NaN,NaN,NaN,0.0,2020-01-01,03:00:00
4,75106001,48.844833,2.338500,0.6,NaN,NaN,NaN,0.0,2020-01-01,04:00:00
...,...,...,...,...,...,...,...,...,...,...
261855,75116008,48.854833,2.233667,4.9,94.0,5.0,NaN,0.0,2024-12-31,19:00:00
261856,75116008,48.854833,2.233667,5.3,93.0,5.1,NaN,0.0,2024-12-31,20:00:00
261857,75116008,48.854833,2.233667,5.6,93.0,4.4,NaN,0.0,2024-12-31,21:00:00
261858,75116008,48.854833,2.233667,5.6,95.0,5.0,NaN,0.0,2024-12-31,22:00:00


Let's check that if all the timeframes are there

In [31]:
meteo75[meteo75["NUM_POSTE"]==75106001]["Heures"].unique()

array([datetime.time(0, 0), datetime.time(1, 0), datetime.time(2, 0),
       datetime.time(3, 0), datetime.time(4, 0), datetime.time(5, 0),
       datetime.time(6, 0), datetime.time(7, 0), datetime.time(8, 0),
       datetime.time(9, 0), datetime.time(10, 0), datetime.time(11, 0),
       datetime.time(12, 0), datetime.time(13, 0), datetime.time(14, 0),
       datetime.time(15, 0), datetime.time(16, 0), datetime.time(17, 0),
       datetime.time(18, 0), datetime.time(19, 0), datetime.time(20, 0),
       datetime.time(21, 0), datetime.time(22, 0), datetime.time(23, 0)],
      dtype=object)

#### We'll keep the rows where the date is between `min(conso["Date"])` and `max(conso["Date"])`

### We can observe that there are multiple stations / data providers.  
There are differents weather stations so there are several duplicate rows for a same date

In [18]:
print("There are", len(meteo75["NUM_POSTE"].unique()), "stations")

There are 6 stations


### We guess that some stations didn't record all the weather data, but that others did.  
Let's calculate the percentage of missing values for each columns and also grouped each station

In [20]:
print("Percentage of NaN per column")
print((meteo75.isnull().mean() * 100).round(1))

print("\nPercentage of NaN per station")
print(meteo75.groupby('NUM_POSTE')[['T', 'U', 'FF', 'PMER', 'RR1']].apply(
    lambda x: x.isnull().mean() * 100).round(1))

Percentage of NaN per column
NUM_POSTE      0.0
LAT            0.0
LON            0.0
AAAAMMJJHH     0.0
T              0.2
U             66.5
FF            50.3
PMER          83.3
RR1           33.3
dtype: float64

Percentage of NaN per station
             T      U     FF   PMER    RR1
NUM_POSTE                                 
75106001   0.0  100.0  100.0  100.0    0.5
75107005   0.9  100.0    0.3  100.0  100.0
75110001   0.0  100.0  100.0  100.0    0.3
75114001   0.0    0.0    0.0    0.0    0.1
75114007   0.0  100.0  100.0  100.0  100.0
75116008   0.1    0.0    0.3  100.0    0.8


We can observe that the station `75114001` has recorded more data than the others

In [13]:
meteo75[meteo75["NUM_POSTE"] == 75114001].isnull().mean() * 100

NUM_POSTE     0.000000
LAT           0.000000
LON           0.000000
AAAAMMJJHH    0.000000
T             0.000000
U             0.000000
FF            0.011403
PMER          0.000000
RR1           0.057015
dtype: float64

We are going to keep the station that has the fewest missing values

#### For the missing values we'll use the interpolation method  